# Monthly Shipment Analysis

Owner: Maya (BI, Surabaya). Last updated: 2025-Q4.

This notebook digs into the previous month's shipments across the three core hubs:
Jakarta, Surabaya, and Makassar. We look at:

1. Volume by hub (origin city).
2. Mean transit time (created -> delivered).
3. Exception rate by route.

**DO NOT** run this against prod-primary. It hits the replica via
`DATABASE_URL_REPLICA`. The aggregation queries are heavy; they'll trip the
DBA pager if pointed at the wrong box.

Caveat: shipment data older than 18 months is in the legacy archive and not
in this dataset. For multi-year retrospectives, talk to Finance IT — they have
a separate Oracle-backed warehouse.

In [ ]:
import os
import pandas as pd
import sqlalchemy as sa

engine = sa.create_engine(os.environ['DATABASE_URL_REPLICA'])

MONTH = '2025-09'

df = pd.read_sql(
    """
    SELECT origin_city, destination_city, status_code, kg, created_at, last_event_at
      FROM shipments_mirror
     WHERE created_at >= date_trunc('month', %(month)s::date)
       AND created_at <  date_trunc('month', %(month)s::date) + interval '1 month'
    """,
    engine,
    params={'month': MONTH + '-01'},
)
df.shape

## Volume by origin hub

Jakarta tends to dominate by sheer volume; Surabaya is the inter-island gateway;
Makassar reflects the eastern Indonesia traffic. The interesting question is the
month-over-month shift, not the absolute split.

In [ ]:
by_hub = (
    df.assign(hub=df['origin_city'].where(
        df['origin_city'].isin(['Jakarta', 'Surabaya', 'Makassar']),
        other='Other',
    ))
    .groupby('hub')
    .size()
    .reset_index(name='shipments')
    .sort_values('shipments', ascending=False)
)
by_hub

## Exception rate by route

An 'exception' is anything in `status_code = 'EXCEPTION'` at end-of-month.
Surabaya → Makassar consistently shows higher exceptions due to ferry delays
in the rainy season. Don't 'fix' it without asking ops first.

TODO(maya): break out by week so we can tease apart weather vs. operational issues.

In [ ]:
by_route = (
    df.assign(is_exception=df['status_code'] == 'EXCEPTION')
    .groupby(['origin_city', 'destination_city'])
    .agg(shipments=('status_code', 'size'), exceptions=('is_exception', 'sum'))
    .assign(exception_rate=lambda d: d['exceptions'] / d['shipments'])
    .sort_values('exception_rate', ascending=False)
    .head(15)
)
by_route